# 24. Ordered navigation paths

**Medium**

Return userid and pages, an array of every page visit ordered by event_seq ascending within the user. Retain repeated pages. event_seq is explicitly supplied because Spark DataFrames have no inherent input order; it is unique per user.


In [0]:
%run ./_setup $exercise_id="24"


**Your solution:** add imports and return your result from `solve`.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window 

def solve(input_df):
    # Write your solution here.
    result_df= (
        input_df
        .withColumn("pages",F.collect_list("page").over(Window.partitionBy("userid").orderBy("event_seq").rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing)))
        .select("userid", "pages").distinct()
    )
    return result_df

# solve(input_df).display()
solve(input_df.orderBy(F.col("event_seq").desc())).display()


In [0]:
def solve(input_df):
    return (
        input_df
        .groupBy("userid")
        .agg(
            F.sort_array(F.collect_list(F.struct("event_seq","page"))).alias("events")
        )
        .select("userid",
                F.transform("events", lambda x: x["page"]).alias("pages")    
            )
    )

solve(input_df).display()

**Check:** run below to test all cases and save progress.


In [0]:
check_report = check_solution(exercise_id, solve, spark)


In [0]:
show_progress(exercise_id, spark)
